In [12]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_squared_error, r2_score

from xgboost import XGBRegressor

# =========================
# 1. Load Data
# =========================
data = pd.read_csv("onion_price_data.csv")

# =========================
# 2. Date Handling
# =========================
data["Arrival_Date"] = pd.to_datetime(
    data["Arrival_Date"], dayfirst=True, errors="coerce"
)

data.sort_values("Arrival_Date", inplace=True)

# =========================
# 3. Remove Extreme Outliers (MANDATORY)
# =========================
for col in ["Min_Price", "Max_Price", "Modal_Price"]:
    q1 = data[col].quantile(0.25)
    q3 = data[col].quantile(0.75)
    iqr = q3 - q1
    data = data[
        (data[col] >= q1 - 1.5 * iqr) &
        (data[col] <= q3 + 1.5 * iqr)
    ]

# =========================
# 4. Lag Features (MOST IMPORTANT)
# =========================
for lag in [1, 7, 14]:
    data[f"Modal_lag_{lag}"] = data["Modal_Price"].shift(lag)

data.dropna(inplace=True)

# =========================
# 5. Log Transform Target
# =========================
y = np.log1p(data[["Min_Price", "Max_Price", "Modal_Price"]])
X = data.drop(columns=["Min_Price", "Max_Price", "Modal_Price"])

# =========================
# 6. Date Features
# =========================
X["Day"] = X["Arrival_Date"].dt.day
X["Month"] = X["Arrival_Date"].dt.month
X["Year"] = X["Arrival_Date"].dt.year
X.drop(columns="Arrival_Date", inplace=True)

# =========================
# 7. Time-Based Split
# =========================
split_date = X["Year"].quantile(0.8)

train_idx = X["Year"] <= split_date
test_idx  = X["Year"] > split_date

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

# =========================
# 8. Encoding
# =========================
categorical_cols = X.select_dtypes(include="object").columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
    ],
    remainder="passthrough"
)

# =========================
# 9. Model
# =========================
xgb = XGBRegressor(
    n_estimators=400,
    learning_rate=0.03,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

model = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("regressor", MultiOutputRegressor(xgb))
    ]
)

# =========================
# 10. Train
# =========================
model.fit(X_train, y_train)

# =========================
# 11. Predict
# =========================
y_pred_log = model.predict(X_test)

# Inverse log
y_pred = np.expm1(y_pred_log)
y_test_actual = np.expm1(y_test)

# =========================
# 12. Evaluation
# =========================
print("\n📊 MODEL PERFORMANCE")

for i, col in enumerate(["Min_Price", "Max_Price", "Modal_Price"]):
    r2 = r2_score(y_test_actual.iloc[:, i], y_pred[:, i])
    rmse = np.sqrt(mean_squared_error(y_test_actual.iloc[:, i], y_pred[:, i]))

    print(f"\n🔹 {col}")
    print(f"R² Score : {r2:.3f}")
    print(f"RMSE     : {rmse:.2f}")

# =========================
# 13. Example Prediction
# =========================
print("\n🔎 SAMPLE PREDICTION")
print("Predicted :", np.round(y_pred[0], 2))
print("Actual    :", np.round(y_test_actual.iloc[0].values, 2))

print("\n✅ Approximate & realistic prediction model ready")
accuracy=model.score(X_test,y_test)
print(f"\nAccuracy: {(accuracy*100):.2f}%")



📊 MODEL PERFORMANCE

🔹 Min_Price
R² Score : 0.436
RMSE     : 266.26

🔹 Max_Price
R² Score : 0.875
RMSE     : 227.83

🔹 Modal_Price
R² Score : 0.898
RMSE     : 185.46

🔎 SAMPLE PREDICTION
Predicted : [ 480.13 1988.78 1791.75]
Actual    : [ 500. 2043. 1900.]

✅ Approximate & realistic prediction model ready

Accuracy: 73.81%
